# GCDG: portable fresh-training component experiment

Copy this **single notebook** into the Denoise-generator repository root, or a subfolder such as `experiments/`. Commit/push the notebook, then pull it on the remote machine. All helper code is embedded; no local `rankgen_gcdg_training.py` file is needed.

Train from scratch, then compare matched held-out conditions at three stages:

1. **Denoiser + supervised heads**, without condition-to-count/histogram controls, using threshold adjacency.
2. **Add the three structural controls**, still using threshold adjacency.
3. **Add constrained decoding**, reusing stage 2 node features and learned edge scores.

No old trained model is loaded. Training and evaluation are enabled, as requested; running all cells will start a fresh experiment. Restoring the best checkpoint created within this fresh run remains enabled. Keep the same kernel for the audit.

**Memory change:** the original dense padded 14-bit node-feature tensor is replaced by sparse node encoding and sparse training-only tail SVD. Dense padding is performed after reduction to 137 columns (9 preserved + 128 components). The 18,000 training graphs, 14-bit hashes, 128-component SVD, losses and model configuration are retained. Sparse/dense numerical differences mean this is not a bitwise reproduction. Condition-vector scaling and control fitting still run on CPU, and no runtime speedup is guaranteed. No automatic mixed precision is enabled.

The experiment concerns synthetic cycle–leg graphs. Stage 1 includes both denoising and learned heads; it is not a diffusion-only or diffusion-free baseline. The three controls are tested together. This is not a molecular valency-repair experiment. Original thesis and generator source files are never edited.


## Baseline checked against the thesis

Defaults follow Appendix Table `tab:synthetic-cycle-leg-proposed-hyperparameters`: **14-bit NSPPK graph conditions; quotient node features; 128-dimensional node-tail SVD; consistency weight 0.3; 100 evaluation steps; 18,000 training graphs and 600 held-out conditions**. Other training parameters come from `ablation1.ipynb` cell 19. The condition vector is not reduced.

**Source discrepancy:** the two inspected notebooks initially assign `nbits=13`, but their executed vectorizer cell overwrites it with **14**, and the saved output confirms 16,384 dimensions. `ablation1_5.ipynb` contains the reported 169 exact/131 strict successes, but its current consistency weight is **0**, conflicting with the thesis baseline **0.3**. `ablation1.ipynb` specifies 0.3 but has different saved counts. This fresh experiment follows the **documented thesis configuration**; its new results must be reported separately. Details are recorded in `BASELINE_AUDIT` below.


In [ ]:
# Install missing dependencies into THIS notebook kernel (not another Python).
# Existing packages are retained. PyTorch is never selected or upgraded here.
import sys, subprocess, importlib, importlib.util, tempfile
from pathlib import Path

if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Select your PyTorch/CUDA kernel first. This cell does not install a torch build.')
import torch
DEPENDENCIES = {
    'pulp': 'PuLP==3.2.2',
    'pytorch_lightning': 'pytorch-lightning==2.5.5',
    'numpy': 'numpy==1.26.4',
    'scipy': 'scipy==1.11.4',
    'pandas': 'pandas==2.3.1',
    'networkx': 'networkx==3.3',
    'sklearn': 'scikit-learn==1.2.2',
    'matplotlib': 'matplotlib==3.10.3',
    'joblib': 'joblib==1.5.1',
    'toolz': 'toolz==1.0.0',
    'dill': 'dill==0.4.0',
    'multiprocessing_on_dill': 'multiprocessing-on-dill==3.5.0a4',
    'threadpoolctl': 'threadpoolctl==3.6.0',
}
missing = [requirement for module, requirement in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
print('Active notebook Python:', sys.executable)
if missing:
    print('Installing missing packages:', ', '.join(missing), flush=True)
    # Constrain torch to the exact installed version, preserving its CUDA/CPU build.
    with tempfile.TemporaryDirectory(prefix='gcdg_dependencies_') as temporary:
        constraint = Path(temporary)/'keep_torch.txt'
        constraint.write_text('torch==' + torch.__version__ + '\n')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                               '--constraint', str(constraint), *missing])
    importlib.invalidate_caches()
import pulp
print('PuLP:', pulp.__version__)
if not pulp.PULP_CBC_CMD(msg=False).available():
    raise RuntimeError('PuLP is installed, but CBC is unavailable. Install the CBC solver on this host.')
print('Dependencies and CBC ready. Training and audit are enabled below.')


In [ ]:
from pathlib import Path
import sys, os, copy, json

# Leave blank when opening this notebook from the repository or a subfolder.
# On an unusual Jupyter launch location, set GCDG_PROJECT_ROOT or this override.
PROJECT_ROOT_OVERRIDE = os.environ.get('GCDG_PROJECT_ROOT', '').strip()

def find_project_root(override=''):
    start = Path(override).expanduser().resolve() if override else Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate/'coco_grape').is_dir():
            return candidate
    raise FileNotFoundError('Open Jupyter inside Denoise-generator, or set PROJECT_ROOT_OVERRIDE '
                            'to that checkout. It must contain coco_grape/.')

PROJECT_ROOT = find_project_root(PROJECT_ROOT_OVERRIDE)
sys.path.insert(0, str(PROJECT_ROOT))
sys.dont_write_bytecode = True

# Outside the checkout by default: git clean/reset in the repo cannot delete these runs.
OUTPUT_ROOT = Path(os.environ.get('GCDG_RUNS_DIR', str(Path.home()/'gcdg_runs'))).expanduser().resolve()
os.environ.setdefault('MPLCONFIGDIR', str(OUTPUT_ROOT/'matplotlib_cache'))

import torch
# CUDA is selected on a CUDA host. Otherwise CPU is selected; MPS is opt-in.
DEVICE = os.environ.get('GCDG_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
TRAINER_CONFIG = dict(accelerator=DEVICE, devices=1, precision='32-true')
CPU_THREADS = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 1)
TRAINING_SEED = 42
RUN_TRAINING = True
RUN_AUDIT = True
PREPARATION_CONFIG = dict(memory_efficient=True, batch_graphs=128)

if DEVICE not in ['cpu', 'cuda', 'mps']:
    raise ValueError('Choose DEVICE="cuda", "cpu", or "mps".')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA requested but unavailable. Install/select a CUDA-enabled PyTorch environment.')
if DEVICE == 'mps' and not torch.backends.mps.is_available():
    raise RuntimeError('MPS requested but unavailable.')
if CPU_THREADS < 1:
    raise ValueError('CPU_THREADS must be positive.')
print('Project:', PROJECT_ROOT)
print('Runs outside Git:', OUTPUT_ROOT)
print('Training device:', DEVICE, '| CPU preparation threads:', CPU_THREADS)
if DEVICE == 'cuda':
    print('CUDA device:', torch.cuda.get_device_name(0))


In [ ]:
# Dataset and representation configuration (thesis baseline)
DATASET_CONFIG = dict(
    cycle_sizes=[3, 4, 5, 6, 7, 8], leg_sizes=[4, 5],
    cycle_labels=[0, 1, 2, 3], leg_labels=[4, 5, 6],
    train_per_cycle=3000, test_per_cycle=100,
    seed=42, max_unique_attempts=20000,
)
GRAPH_CONFIG = dict(radius=1, distance=4, connector=1, nbits=14,
                    dense=True, parallel=True, use_edges_as_features=True)
NODE_CONFIG = dict(mode='quotient', nbits=14, n_jobs=-1)


In [ ]:
# Training hyperparameters: edit here before training
DENOISE_CONFIG = {'latent_embedding_dimension': 128,
 'number_of_transformer_layers': 4,
 'transformer_attention_head_count': 8,
 'transformer_dropout': 0.05,
 'learning_rate': 0.0003,
 'maximum_epochs': 5000,
 'batch_size': 32,
 'total_steps': 50,
 'verbose': True,
 'degree_temperature': None,
 'balance_degree_loss': False,
 'degree_class_weight_cap': 2,
 'lambda_edge_importance': 5,
 'lambda_clean_edge_importance': 0,
 'balance_edge_loss': True,
 'lambda_label_importance': 140,
 'lambda_edge_label_importance': 0,
 'lambda_degree_importance': 60,
 'lambda_node_exist_importance': 5,
 'lambda_consistency': 0.3,
 'lambda_distance_importance': 1,
 'lambda_recon_importance': 0,
 'lambda_x0_importance': 10,
 'lambda_condition_x0_importance': 0,
 'condition_x0_sampling_blend': 0,
 'max_distance_class': 3,
 'label_feature_index': 2,
 'noise_label_factor': 10,
 'noise_degree_factor': 10,
 'sigma_min': 0.01,
 'sigma_max': 0.1,
 'sampling_final_sigma': 0.0,
 'use_guidance': False,
 'early_stopping': True,
 'early_stop_metric': 'val_total',
 'early_stop_mode': 'min',
 'early_stop_patience': 100,
 'early_stop_min_delta': 1e-05,
 'restore_best_weights': True,
 'checkpoint_policy': 'best',
 'use_dim_reduction': True,
 'dim_reduction_method': 'svd',
 'dim_reduction_components': 128,
 'dim_reduction_keep_prefix': 'auto',
 'pca_whiten': False,
 'use_condition_dim_reduction': False,
 'condition_dim_reduction_method': 'svd',
 'condition_dim_reduction_components': 128,
 'condition_dim_reduction_keep_prefix': 2,
 'denoise_discrete_channels': False,
 'discrete_diffusion_mode': 'absorbing',
 'row_embedding_scale': 0.0,
 'discrete_projection_sigma_threshold': 0,
 'use_joint_edge_denoising': False,
 'joint_edge_message_passing_steps': 3,
 'joint_edge_delete_prob': 0.1,
 'joint_edge_add_prob': 0.02,
 'joint_edge_update_momentum': 0.7,
 'joint_edge_threshold': 0.4,
 'random_row_permutation': False,
 'use_matched_row_loss': False,
 'project_existence_during_sampling': True,
 'project_degree_during_sampling': True,
 'project_label_during_sampling': True,
 'use_condition_node_count': True,
 'use_condition_degree_histogram': True,
 'use_condition_label_histogram': True,
 'balance_label_loss': True,
 'label_class_weight_cap': 2}

DECODER_CONFIG = {'verbose': True,
 'non_edges_factor': 4,
 'num_augmentation_iterations': 0,
 'augmentation_noise': 0,
 'enforce_connectivity': True,
 'warm_start_mst': True,
 'existence_threshold': 0.5,
 'degree_slack_penalty': 20}

PIPELINE_CONFIG = {'use_edge_supervision': True,
 'use_edge_fraction': 0.3,
 'use_distance_supervision': True,
 'distance_edge_blend': 0,
 'edge_generation_mode': 'direct',
 'edge_diffusion_generator': None,
 'verbose': True,
 'use_direct_svd_encodings': False,
 'direct_node_svd_components': 64,
 'direct_graph_svd_components': 64,
 'direct_node_svd_keep_prefix': 3,
 'direct_graph_svd_keep_prefix': 2,
 'direct_svd_random_state': 42}


In [ ]:
# Evaluation configuration. 100 conditions per cycle matches the thesis sample count.
AUDIT_CONFIG = dict(
    sampling_steps=100,
    sampling_seeds=[0],          # use [0, 1, 2] for repeated sampling
    batch_size=10,
    per_cycle=100,
    selection_seed=2026,
    edge_threshold=0.5,
    enforce_connectivity=True,
    degree_slack_penalty=20,
    solver_seconds_per_graph=60,
    leg_sizes=DATASET_CONFIG['leg_sizes'],
)
# Optional short PILOT; uncomment BEFORE training. Do not label a pilot as the thesis baseline.
# DATASET_CONFIG.update(train_per_cycle=100, test_per_cycle=10)
# DENOISE_CONFIG.update(maximum_epochs=25, early_stop_patience=10)
# AUDIT_CONFIG.update(per_cycle=10, sampling_seeds=[0])


In [ ]:
BASELINE_AUDIT = {
    'authority': 'thesis appendix, tab:synthetic-cycle-leg-proposed-hyperparameters',
    'source_notebooks': ['ablation1.ipynb', 'ablation1_5.ipynb'],
    'graph_nbits': 14, 'node_nbits': 14, 'lambda_consistency': 0.3,
    'training_total_steps': 50, 'evaluation_steps': 100,
    'train_graphs': 18000, 'evaluation_graphs': 600,
    'discrepancies': [
        'nbits=13 is overwritten by nbits=14 in vectorizer cell 15; saved dimensions confirm 16384.',
        'ablation1_5 currently sets consistency=0 but contains the thesis counts 169 exact/131 strict; '
        'the thesis baseline says 0.3. The old execution weight cannot be verified from saved output.',
        'ablation1 sets consistency=0.3 but its saved counts differ. New defaults follow the documented thesis.',
    ],
    'implementation_change': 'Sparse quotient features and sparse training-only tail SVD before dense padding; '
                             'full dataset, hash bits, SVD dimensions, prefix, losses and architecture retained.',
    'reproduction_claim': 'Fresh experiment, not bitwise reproduction of historical counts.',
}
print('Thesis baseline selected; historical discrepancy and sparse preparation are recorded.')


## Utilities

The next cell defines data generation, fresh training and the paired evaluation. It does not train or load a model. Leave it unchanged unless you are modifying the experiment logic.


In [ ]:
"""Synthetic data functions copied from ablation1_5.ipynb cell 11; no notebook execution."""
import random
import numpy as np
import networkx as nx
from collections import Counter

def make_cycle_leg_graph(cycle_size, leg_size, seed, graph_id, split):
    """Create one labeled cycle with one attached path leg."""
    cycle_size = int(cycle_size)
    leg_size = int(leg_size)
    if cycle_size < 3:
        raise ValueError("cycle_size must be at least 3")
    if leg_size < 1:
        raise ValueError("leg_size must be positive")

    rng = random.Random(int(seed))
    graph = nx.Graph()

    cycle_nodes = list(range(cycle_size))
    leg_nodes = list(range(cycle_size, cycle_size + leg_size))

    for index, node in enumerate(cycle_nodes):
        label = rng.choice(MOTIF_LABELS["target"])
        graph.add_node(
            node,
            label=label,
            attr=label,
            plot_label=label,
            display_label=label,
            plot_color=LABEL_TO_COLOR[label],
            motif="target",
            motif_role="target",
            motif_type="cycle",
            node_role="cycle",
            cycle_position=index,
        )

    for index, node in enumerate(leg_nodes):
        label = rng.choice(MOTIF_LABELS["connector"])
        graph.add_node(
            node,
            label=label,
            attr=label,
            plot_label=label,
            display_label=label,
            plot_color=LABEL_TO_COLOR[label],
            motif="connector",
            motif_role="connector",
            motif_type="path",
            node_role="leg",
            leg_position=index,
        )

    for index, node in enumerate(cycle_nodes):
        graph.add_edge(
            node,
            cycle_nodes[(index + 1) % cycle_size],
            label=0,
            edge_attr=0,
            plot_label=0,
            plot_color="#555555",
            motif="target",
            motif_role="target",
            motif_type="cycle",
        )

    # Random attachment position prevents node index from identifying the category.
    attachment_node = rng.choice(cycle_nodes)
    graph.add_edge(
        attachment_node,
        leg_nodes[0],
        label=0,
        edge_attr=0,
        plot_label=0,
        plot_color="#555555",
        motif="connector",
        motif_role="connector",
        motif_type="path",
    )
    for left, right in zip(leg_nodes[:-1], leg_nodes[1:]):
        graph.add_edge(
            left,
            right,
            label=0,
            edge_attr=0,
            plot_label=0,
            plot_color="#555555",
            motif="connector",
            motif_role="connector",
            motif_type="path",
        )

    graph.graph.update(
        {
            "graph_id": int(graph_id),
            "split": str(split),
            "category": f"cycle_{cycle_size}",
            "motif_family": "cycle_with_leg",
            "target_type": "cycle",
            "connector_type": "path",
            "context_type": None,
            "target_size": cycle_size,
            "cycle_size": cycle_size,
            "connector_size": leg_size,
            "leg_size": leg_size,
            "context_size": 0,
            "branch_count": 1,
            "branch_lengths": (leg_size,),
            "total_nodes": cycle_size + leg_size,
            "attachment_node": attachment_node,
        }
    )
    return graph

def labeled_cycle_leg_signature(graph):
    """Canonical label signature, invariant to reflecting the rooted cycle."""
    cycle_size = int(graph.graph["cycle_size"])
    leg_size = int(graph.graph["leg_size"])
    attachment = int(graph.graph["attachment_node"])

    forward_cycle = tuple(
        int(graph.nodes[(attachment + offset) % cycle_size]["label"])
        for offset in range(cycle_size)
    )
    reverse_cycle = tuple(
        int(graph.nodes[(attachment - offset) % cycle_size]["label"])
        for offset in range(cycle_size)
    )
    canonical_cycle = min(forward_cycle, reverse_cycle)
    leg_labels = tuple(
        int(graph.nodes[cycle_size + offset]["label"])
        for offset in range(leg_size)
    )
    return cycle_size, leg_size, canonical_cycle, leg_labels

def build_balanced_cycle_leg_split(
    per_category,
    split,
    seed,
    used_signatures=None,
):
    """Build a balanced split with no repeated labeled motif signatures."""
    graphs = []
    targets = []
    rng = random.Random(int(seed))
    graph_id = 0
    used_signatures = set() if used_signatures is None else used_signatures
    split_signatures = set()

    for cycle_size in CYCLE_SIZES:
        leg_schedule = [
            LEG_SIZES[index % len(LEG_SIZES)]
            for index in range(int(per_category))
        ]
        rng.shuffle(leg_schedule)

        category_graphs = []
        for leg_size in leg_schedule:
            graph = None
            signature = None
            for _ in range(MAX_UNIQUE_ATTEMPTS_PER_GRAPH):
                graph_seed = rng.randint(0, 2**31 - 1)
                candidate = make_cycle_leg_graph(
                    cycle_size=cycle_size,
                    leg_size=leg_size,
                    seed=graph_seed,
                    graph_id=graph_id,
                    split=split,
                )
                candidate_signature = labeled_cycle_leg_signature(candidate)
                if candidate_signature not in used_signatures:
                    graph = candidate
                    signature = candidate_signature
                    break
            if graph is None:
                raise RuntimeError(
                    "Could not create enough unique labeled graphs for "
                    f"cycle={cycle_size}, leg={leg_size}. Increase the target/leg "
                    "alphabet sizes or reduce the requested graph count."
                )
            used_signatures.add(signature)
            split_signatures.add(signature)
            graph.graph["labeled_signature"] = signature
            category_graphs.append((graph, f"cycle_{cycle_size}"))
            graph_id += 1

        rng.shuffle(category_graphs)
        graphs.extend(graph for graph, _ in category_graphs)
        targets.extend(target for _, target in category_graphs)

    paired = list(zip(graphs, targets))
    rng.shuffle(paired)
    return (
        [graph for graph, _ in paired],
        [target for _, target in paired],
        split_signatures,
    )

def ensure_plot_attrs(graphs):
    for graph in graphs:
        for _, data in graph.nodes(data=True):
            label = int(data.get("label", 0))
            data["label"] = label
            data["attr"] = label
            data["plot_label"] = label
            data["display_label"] = label
            data["plot_color"] = LABEL_TO_COLOR.get(label, "#cccccc")
        for _, _, data in graph.edges(data=True):
            data["label"] = 0
            data["edge_attr"] = 0
            data["plot_label"] = 0
            data["plot_color"] = "#555555"

def make_dataset(cfg):
    global CYCLE_SIZES, LEG_SIZES, MOTIF_LABELS, LABEL_TO_COLOR, MAX_UNIQUE_ATTEMPTS_PER_GRAPH
    CYCLE_SIZES = tuple(cfg['cycle_sizes'])
    LEG_SIZES = tuple(cfg['leg_sizes'])
    MOTIF_LABELS = dict(target=list(cfg['cycle_labels']), connector=list(cfg['leg_labels']))
    if set(MOTIF_LABELS['target']) & set(MOTIF_LABELS['connector']):
        raise ValueError('Cycle-role and leg-role labels must be disjoint.')
    LABEL_TO_COLOR = {label: '#d62828' for label in MOTIF_LABELS['target']}
    LABEL_TO_COLOR.update({label: '#1f77b4' for label in MOTIF_LABELS['connector']})
    MAX_UNIQUE_ATTEMPTS_PER_GRAPH = int(cfg.get('max_unique_attempts', 20000))
    if any(int(cfg[k]) % len(LEG_SIZES) for k in ['train_per_cycle', 'test_per_cycle']):
        raise ValueError('Counts per cycle must be divisible by the number of leg sizes.')
    used = set()
    train, train_targets, train_sig = build_balanced_cycle_leg_split(
        int(cfg['train_per_cycle']), 'train', int(cfg['seed']), used)
    test, test_targets, test_sig = build_balanced_cycle_leg_split(
        int(cfg['test_per_cycle']), 'test', int(cfg['seed']) + 1, used)
    ensure_plot_attrs(train)
    ensure_plot_attrs(test)
    assert train_sig.isdisjoint(test_sig)
    return train, train_targets, test, test_targets


"""Sparse preparation for the documented quotient/SVD GCDG configuration.

Node hashes, column order and loss settings are retained. Sparse SVD may differ
numerically from dense SVD; this is a fresh experiment, not bitwise reproduction.
No existing source module or file is changed.
"""
import time
from types import MethodType
import numpy as np
from scipy import sparse
from sklearn.decomposition import TruncatedSVD


def baseline_quotient_decomposition():
    from coco_grape.module.quotientgraph.operator import (
        add, compose, combination, node, prepend_label, cycle, tree, path,
        neighborhood, unlabel)
    distances = add(*[compose(combination(distance=(d, d)), node(), prepend_label(label=d))
                      for d in range(2, 10)])
    labelled = add(distances, cycle(), tree(), path(number_of_edges=(1, 3)),
                   neighborhood(radius=(0, 2)))
    return add(labelled, compose(labelled, unlabel()))


def inject_sparse_labels(encodings, graphs, label_to_idx=None):
    # Same first-encounter string mapping and insertion at column 2 as source.
    if label_to_idx is None:
        labels = []
        for graph in graphs:
            for _, attrs in graph.nodes(data=True):
                label = attrs.get('label', '__UNLABELED__')
                if label is None or (isinstance(label, str) and not label.strip()):
                    label = '__UNLABELED__'
                labels.append(str(label))
        label_to_idx = {label: i for i, label in enumerate(dict.fromkeys(labels))}
    label_to_idx = {str(k): int(v) for k, v in label_to_idx.items()}
    default = label_to_idx.get('__UNLABELED__', next(iter(label_to_idx.values()), 0))
    out = []
    for x, graph in zip(encodings, graphs):
        x = sparse.csr_matrix(x, dtype=np.float64)
        if x.shape[0] != len(graph) or x.shape[1] < 2:
            raise ValueError('Unexpected quotient feature shape.')
        ids = []
        for _, attrs in graph.nodes(data=True):
            label = attrs.get('label', '__UNLABELED__')
            if label is None or (isinstance(label, str) and not label.strip()):
                label = '__UNLABELED__'
            ids.append(label_to_idx.get(str(label), default))
        labels = sparse.csr_matrix((np.ones(len(ids)), (np.arange(len(ids)), ids)),
                                   shape=(len(ids), len(label_to_idx)))
        out.append(sparse.hstack([x[:, :2], labels, x[:, 2:]], format='csr'))
    return out, label_to_idx


def sparse_generator_class(base):
    """Keep original setup/fit/losses; feed them reduced training arrays only."""
    class SparsePreparedGenerator(base):
        def _reduce_sparse_training(self, matrices):
            out = []
            prefix = self.dim_reduction_keep_prefix
            for start in range(0, len(matrices), self.sparse_batch_graphs):
                batch = matrices[start:start+self.sparse_batch_graphs]
                sizes = [x.shape[0] for x in batch]
                rows = sparse.vstack(batch, format='csr')
                values = np.hstack([rows[:, :prefix].toarray(),
                                    self.reducer.transform(rows[:, prefix:])])
                out.extend(np.split(values, np.cumsum(sizes)[:-1]))
            return out

        def setup(self, node_encodings_list, **kwargs):
            if not all(sparse.issparse(x) for x in node_encodings_list):
                raise ValueError('Memory-efficient setup requires sparse node features.')
            if not self.use_dim_reduction or self.dim_reduction_method != 'svd':
                raise ValueError('Memory-efficient mode requires the baseline node-tail SVD.')
            label_width = int(self._node_label_one_hot_width)
            if getattr(self, '_auto_dim_reduction_keep_prefix', False):
                self.dim_reduction_keep_prefix = 2 + label_width
            prefix = int(self.dim_reduction_keep_prefix)
            started = time.perf_counter()
            print('Sparse node-tail SVD: stacking real training rows (no padded dense array).', flush=True)
            rows = sparse.vstack(node_encodings_list, format='csr')
            self._sparse_raw_width = rows.shape[1]
            self._orig_tail_dim = rows.shape[1] - prefix
            tail = rows[:, prefix:]
            count = max(1, min(int(self.dim_reduction_components),
                               max(1, tail.shape[0]-1), tail.shape[1]))
            if count >= tail.shape[1]:
                raise ValueError('Choose fewer SVD components than hashed tail columns.')
            print(f'Sparse SVD: {tail.shape}, {tail.nnz:,} nonzeros -> {count} components.', flush=True)
            self.reducer = TruncatedSVD(n_components=count, random_state=self.random_state)
            self.reducer.fit(tail)
            self._sparse_reduced_width = prefix + count
            del rows, tail
            reduced = self._reduce_sparse_training(node_encodings_list)
            print(f'Sparse SVD ready in {time.perf_counter()-started:.1f}s; dense width '
                  f'{self._sparse_raw_width} -> {self._sparse_reduced_width}.', flush=True)
            # Original code now pads/scales only the reduced features. The fitted
            # reducer remains attached for raw inference and inverse reconstruction.
            return super().setup(node_encodings_list=reduced, **kwargs)

        def _fit_reducer(self, x):
            if x.shape[-1] != self._sparse_reduced_width or self.reducer is None:
                raise ValueError('Unexpected feature width during sparse preparation.')
            # Already fitted on all real training rows above; do not fit it again.

        def _apply_reducer(self, x):
            if x.shape[-1] == self._sparse_reduced_width:
                return x
            if x.shape[-1] != self._sparse_raw_width:
                raise ValueError('Unexpected inference feature width.')
            return super()._apply_reducer(x)

        def fit(self, node_encodings_list, **kwargs):
            print('Training setup: reduced features ready; fitting condition controls on CPU.', flush=True)
            reduced = self._reduce_sparse_training(node_encodings_list)
            return super().fit(node_encodings_list=reduced, **kwargs)
    return SparsePreparedGenerator


def install_sparse_pipeline(pipeline, batch_graphs=128):
    """Attach sparse encoding and per-graph dense edge sampling to this instance."""
    pipeline.node_graph_vectorizer.return_dense = False
    generator = pipeline.conditioning_to_node_embeddings_generator.conditional_node_generator
    generator.sparse_batch_graphs = int(batch_graphs)
    if generator.sparse_batch_graphs < 1:
        raise ValueError('batch_graphs must be positive.')

    def encode(this, graphs):
        values = []
        for start in range(0, len(graphs), batch_graphs):
            values.extend(this.node_graph_vectorizer.transform(graphs[start:start+batch_graphs]))
            done = min(start+batch_graphs, len(graphs))
            if start == 0 or done == len(graphs) or done % (batch_graphs*8) == 0:
                print(f'Sparse node encoding: {done}/{len(graphs)} graphs', flush=True)
        values, mapping = inject_sparse_labels(values, graphs, getattr(this, '_label_to_idx', None))
        this._label_to_idx = mapping
        this._idx_to_label = {v: k for k, v in mapping.items()}
        this._node_label_one_hot_width = len(mapping)
        decoder = this.node_embeddings_to_graph_generator
        decoder._idx_to_label = this._idx_to_label
        decoder._node_label_one_hot_width = len(mapping)
        generator._idx_to_label = this._idx_to_label
        generator._node_label_one_hot_width = len(mapping)
        conditions = this.graph_encode(graphs)
        decoder._last_conditioning_vectors = list(conditions)
        return values, conditions

    decoder = pipeline.node_embeddings_to_graph_generator
    original_targets = decoder.adj_mtx_to_targets

    def targets(this, adj_mtx_list, node_encodings_list, use_edge_fraction,
                force_bi_directional_edges=True, is_training=False):
        all_targets, all_pairs = [], []
        for gi, (adj, enc) in enumerate(zip(adj_mtx_list, node_encodings_list)):
            # Only one small graph is dense; original distance/order calculations
            # and global random subset selection are retained.
            dense = enc.toarray() if sparse.issparse(enc) else np.asarray(enc)
            y, pairs = original_targets([adj], [dense], use_edge_fraction=1.0,
                                        force_bi_directional_edges=force_bi_directional_edges,
                                        is_training=False)
            all_targets.extend(y.tolist())
            all_pairs.extend((gi, i, j) for _, i, j in pairs)
        if is_training and use_edge_fraction < 1:
            count = int(round(len(all_pairs)*use_edge_fraction))
            indices = np.random.choice(len(all_pairs), count, replace=False)
            all_targets = [all_targets[i] for i in indices]
            all_pairs = [all_pairs[i] for i in indices]
        print(f'Edge supervision ready: {len(all_pairs):,} directed pairs.', flush=True)
        return np.asarray(all_targets), all_pairs

    pipeline.encode = MethodType(encode, pipeline)
    def preserve_sparse(this, values):
        if this.use_direct_svd_encodings:
            raise ValueError('Direct SVD encodings must remain disabled for this sparse baseline path.')
        return values
    # fit() calls this again after encode(); avoid its unconditional dense cast.
    pipeline._transform_direct_node_encodings = MethodType(preserve_sparse, pipeline)
    decoder.adj_mtx_to_targets = MethodType(targets, decoder)
    return pipeline


"""Fresh-training GCDG experiment and paired inference-stage attribution.

No model is loaded from a previous experiment. Imports of the user's generator
are deferred until make_pipeline/fit_fresh is explicitly called.
"""
from pathlib import Path
from contextlib import contextmanager
from collections import Counter
from datetime import datetime, timezone
import copy
import hashlib
import inspect
import json
import os
import random
import time
import platform
import subprocess
import importlib.metadata
import numpy as np
import pandas as pd
import networkx as nx
import torch


def dump_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, default=str) + '\n')


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


def weight_hash(module):
    digest = hashlib.sha256()
    for key, tensor in sorted(module.state_dict().items()):
        digest.update(key.encode())
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


@contextmanager
def overrides(obj, **values):
    missing = object()
    old = {key: obj.__dict__.get(key, missing) for key in values}
    for key, value in values.items():
        setattr(obj, key, value)
    try:
        yield
    finally:
        for key, value in old.items():
            if value is missing:
                obj.__dict__.pop(key, None)
            else:
                setattr(obj, key, value)


def make_pipeline(graph_cfg, node_cfg, denoise_cfg, decoder_cfg, pipeline_cfg, run, preparation_cfg=None):
    from coco_grape.data_processor.generative.conditional_denoising_node_generator import ConditionalNodeGenerator
    from coco_grape.data_processor.generative.decompositional_encoder_decoder import (
        ConditionalNodeGeneratorModel, DecompositionalNodeEncoderDecoder, DecompositionalEncoderDecoder)
    from coco_grape.graph_vectorizer.nsppk import NSPPK
    from coco_grape.module.quotientgraph.vectorize import QuotientGraphNodeTransformer
    preparation_cfg = dict(preparation_cfg or dict(memory_efficient=True, batch_graphs=128))
    efficient = bool(preparation_cfg['memory_efficient'])
    graph = NSPPK(**graph_cfg)
    mode = node_cfg['mode']
    if mode == 'quotient':
        node = QuotientGraphNodeTransformer(decomposition_function=baseline_quotient_decomposition(),
                                           nbits=node_cfg['nbits'], n_jobs=node_cfg.get('n_jobs', 1),
                                           return_dense=not efficient)
    else:
        raise ValueError('This experiment uses the thesis quotient node representation; set mode="quotient".')
    params = copy.deepcopy(denoise_cfg)
    params['checkpoint_dir'] = str(Path(run).resolve() / 'checkpoints')
    unsupported = set(params) - set(inspect.signature(ConditionalNodeGenerator.__init__).parameters)
    if unsupported:
        raise ValueError(f'Unsupported generator parameters in this source version: {unsupported}')
    generator_class = sparse_generator_class(ConditionalNodeGenerator) if efficient else ConditionalNodeGenerator
    generator = generator_class(**params)
    provider = ConditionalNodeGeneratorModel(conditional_node_generator=generator, verbose=params.get('verbose', True))
    decoder = DecompositionalNodeEncoderDecoder(adjacency_matrix_classifier=None,
                                               node_label_classifier=None, edge_label_classifier=None,
                                               **decoder_cfg)
    pipeline = DecompositionalEncoderDecoder(graph_vectorizer=graph, node_graph_vectorizer=node,
                                            conditioning_to_node_embeddings_generator=provider,
                                            node_embeddings_to_graph_generator=decoder, **pipeline_cfg)
    if efficient:
        if pipeline_cfg.get('use_direct_svd_encodings') or pipeline_cfg.get('edge_generation_mode') != 'direct':
            raise ValueError('Sparse preparation supports the documented direct-edge baseline, without direct SVD.')
        if not params.get('use_dim_reduction') or params.get('dim_reduction_method') != 'svd':
            raise ValueError('Sparse preparation requires node-tail SVD enabled.')
        if any(x is not None for x in [decoder.adjacency_matrix_classifier, decoder.node_label_classifier,
                                      decoder.edge_label_classifier]) or decoder_cfg.get('num_augmentation_iterations', 0):
            raise ValueError('Sparse preparation supports the baseline without separate reconstruction classifiers/augmentation.')
        install_sparse_pipeline(pipeline, batch_graphs=int(preparation_cfg['batch_graphs']))
    return pipeline


def fit_fresh(project_root, output_root, dataset_cfg, graph_cfg, node_cfg, denoise_cfg,
              decoder_cfg, pipeline_cfg, trainer_cfg, training_seed=42, cpu_threads=2, preparation_cfg=None):
    """Create a new random-initialised pipeline and train it; never load a model."""
    accelerator = trainer_cfg.get('accelerator', 'cpu')
    if accelerator not in ['cpu', 'cuda', 'mps']:
        raise ValueError('Select cpu, cuda, or mps explicitly.')
    if accelerator == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA requested but unavailable; refusing to run a CPU fallback.')
    if accelerator == 'mps' and not torch.backends.mps.is_available():
        raise RuntimeError('MPS requested but unavailable; refusing to run a CPU fallback.')
    preparation_cfg = dict(preparation_cfg or dict(memory_efficient=True, batch_graphs=128))
    run = Path(output_root).expanduser().resolve() / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    run.mkdir(parents=True, exist_ok=False)
    seed_all(training_seed)
    torch.set_num_threads(cpu_threads)
    config = dict(dataset=dataset_cfg, graph=graph_cfg, node=node_cfg, denoiser=denoise_cfg,
                  decoder=decoder_cfg, pipeline=pipeline_cfg, trainer=trainer_cfg,
                  preparation=preparation_cfg, cpu_threads=cpu_threads,
                  training_seed=training_seed, from_scratch=True, old_model_loaded=False)
    dump_json(run / 'training_config.json', config)
    source_files = ['coco_grape/data_processor/generative/conditional_denoising_node_generator.py',
                    'coco_grape/data_processor/generative/decompositional_encoder_decoder.py']
    dump_json(run / 'source_hashes.json', {p: hashlib.sha256((Path(project_root)/p).read_bytes()).hexdigest()
                                         for p in source_files})
    try:
        revision = subprocess.check_output(['git', '-C', str(project_root), 'rev-parse', 'HEAD'],
                                           text=True, stderr=subprocess.DEVNULL).strip()
    except (OSError, subprocess.CalledProcessError):
        revision = None
    packages = {}
    for name in ['torch', 'pytorch-lightning', 'numpy', 'scipy', 'scikit-learn', 'networkx', 'pulp']:
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    dump_json(run / 'environment.json', dict(python=platform.python_version(), platform=platform.platform(),
        git_revision=revision, packages=packages, accelerator=accelerator,
        cuda_available=torch.cuda.is_available(),
        cuda_device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        torch_threads=torch.get_num_threads()))
    print('New run:', run, flush=True)
    print('Creating fresh train/held-out graphs...', flush=True)
    train, train_targets, test, test_targets = make_dataset(dataset_cfg)
    pipeline = make_pipeline(graph_cfg, node_cfg, denoise_cfg, decoder_cfg, pipeline_cfg, run, preparation_cfg)
    pipeline.conditioning_to_node_embeddings_generator.conditional_node_generator.device = torch.device(accelerator)
    import coco_grape.data_processor.generative.conditional_denoising_node_generator as source
    original_trainer = source.pl.Trainer
    from pytorch_lightning.loggers import CSVLogger
    from threadpoolctl import threadpool_limits
    # Apply user-selected CPU/MPS/CUDA Trainer options in memory. Original files
    # are untouched. Every log/checkpoint belongs to this new run directory.
    class ExperimentTrainer(original_trainer):
        def __init__(self, *args, **kwargs):
            kwargs.update(trainer_cfg)
            kwargs['default_root_dir'] = str(run)
            kwargs['logger'] = CSVLogger(save_dir=str(run), name='training_logs')
            kwargs['enable_progress_bar'] = True
            kwargs['log_every_n_steps'] = 10
            super().__init__(*args, **kwargs)
    cwd = Path.cwd()
    start = time.perf_counter()
    try:
        os.chdir(run)
        with overrides(source.pl, Trainer=ExperimentTrainer), threadpool_limits(limits=cpu_threads):
            pipeline.fit(train, train_conditioning_to_node_embeddings_generator=True,
                         train_node_embeddings_to_graph_generator=True)
    finally:
        os.chdir(cwd)
    receipt = dict(from_scratch=True, old_model_loaded=False, run=str(run),
                   training_graphs=len(train), heldout_graphs=len(test),
                   fit_seconds=time.perf_counter()-start, training_seed=training_seed)
    dump_json(run / 'training_receipt.json', receipt)
    print('New model trained from scratch:', run)
    return pipeline, test, test_targets, run, receipt


def choose_conditions(graphs, targets, cycles, per_cycle, seed):
    rng = np.random.default_rng(seed)
    indices = []
    for cycle in cycles:
        ids = [i for i, target in enumerate(targets) if str(target) in {str(cycle), f'cycle_{cycle}'}]
        if len(ids) < per_cycle:
            raise ValueError(f'Only {len(ids)} held-out graphs available for cycle {cycle}.')
        indices.extend(rng.choice(ids, size=per_cycle, replace=False).tolist())
    return [graphs[i] for i in indices], [int(str(targets[i]).split('_')[-1]) for i in indices], indices


def canonical_label(value):
    try:
        x = float(value)
        return int(x) if x.is_integer() else value
    except (TypeError, ValueError):
        return value


def graph_metrics(graph, reference, requested, cycle_labels, leg_labels, encoding=None, leg_sizes=(4, 5)):
    if graph is None:
        return dict(decoded=False, connected=False, exact_cycle=False, strict=False,
                    cycle_leg_shape=False, node_count_error=np.nan, degree_hist_l1=np.nan,
                    label_hist_l1=np.nan, predicted_degree_mae=np.nan)
    cycles = nx.cycle_basis(graph)
    unique = len(cycles) == 1
    nodes = set(cycles[0]) if unique else set()
    exact = unique and len(nodes) == requested
    labels_ok = unique and all(canonical_label(data.get('label')) in
                              (set(cycle_labels) if u in nodes else set(leg_labels))
                              for u, data in graph.nodes(data=True))
    connected = graph.number_of_nodes() > 0 and nx.is_connected(graph)
    rest = set(graph) - nodes
    leg = graph.subgraph(rest)
    crossing = [(u, v) for u, v in graph.edges() if (u in nodes) != (v in nodes)]
    shape = (unique and connected and len(rest) in set(leg_sizes) and len(crossing) == 1
             and nx.is_tree(leg) and max(dict(leg.degree()).values(), default=0) <= 2)
    if shape and len(rest) > 1:
        attach = next(u if u in rest else v for u, v in crossing)
        shape = leg.degree(attach) == 1
    degree_a = Counter(dict(graph.degree()).values())
    degree_b = Counter(dict(reference.degree()).values())
    labels_a = Counter(canonical_label(x.get('label')) for _, x in graph.nodes(data=True))
    labels_b = Counter(canonical_label(x.get('label')) for _, x in reference.nodes(data=True))
    pred_mae = np.nan
    if encoding is not None and len(graph):
        pred_mae = float(np.mean([abs(graph.degree(u)-encoding[int(u), 1]) for u in graph]))
    return dict(decoded=True, connected=connected, exact_cycle=exact, strict=bool(exact and labels_ok),
                cycle_leg_shape=bool(shape), node_count_error=abs(len(graph)-len(reference)),
                degree_hist_l1=sum(abs(degree_a[x]-degree_b[x]) for x in degree_a.keys() | degree_b.keys()),
                label_hist_l1=sum(abs(labels_a[x]-labels_b[x]) for x in labels_a.keys() | labels_b.keys()),
                predicted_degree_mae=pred_mae)


def threshold_adjacencies(encodings, scores, threshold, existence_threshold):
    out = []
    for encoding, matrix in zip(encodings, scores):
        p = .5 * (np.asarray(matrix) + np.asarray(matrix).T)
        active = np.asarray(encoding)[:, 0] >= existence_threshold
        a = ((p >= threshold) & active[:, None] & active[None, :]).astype(int)
        np.fill_diagonal(a, 0)
        out.append(a)
    return out


@contextmanager
def capture_generation(module, captured):
    original = module.generate
    def generate(*args, **kwargs):
        result = original(*args, **kwargs)
        captured['tensor'] = result.detach().clone()
        captured['cache'] = {k: clone_cache(v) for k, v in module.__dict__.items() if k.startswith('_last')}
        captured['calls'] = captured.get('calls', 0) + 1
        return result
    with overrides(module, generate=generate):
        yield


def clone_cache(value):
    if torch.is_tensor(value):
        return value.detach().clone()
    if isinstance(value, np.ndarray):
        return value.copy()
    if isinstance(value, dict):
        return {k: clone_cache(v) for k, v in value.items()}
    if isinstance(value, list):
        return [clone_cache(v) for v in value]
    if isinstance(value, tuple):
        return tuple(clone_cache(v) for v in value)
    return copy.deepcopy(value)


@contextmanager
def replay_generation(module, captured):
    """Return the identical neural sample; do not run the sampler a second time."""
    values = dict(captured['cache'])
    values['generate'] = lambda *a, **kw: captured['tensor'].detach().clone()
    with overrides(module, **values):
        yield


def probability_hash(scores):
    h = hashlib.sha256()
    for matrix in scores:
        h.update(np.asarray(matrix, dtype=np.float64).tobytes())
    return h.hexdigest()


def decode_stage(pipeline, conditions, encodings, scores, constrained, cfg):
    """Clone the decoder, preserving model state and the paired edge scores."""
    decoder = copy.copy(pipeline.node_embeddings_to_graph_generator)
    encodings = [np.asarray(x).copy() for x in encodings]
    decoder.use_molecular_valency_projection = False
    decoder._last_conditioning_vectors = list(conditions)
    matrices = [np.asarray(p).copy() for p in scores]
    if constrained:
        decoder.enforce_connectivity = bool(cfg['enforce_connectivity'])
        decoder.degree_slack_penalty = cfg['degree_slack_penalty']
        original = decoder.optimize_adjacency_matrix
        def optimise(probabilities, degrees):
            return original(probabilities, degrees, timeLimit=int(cfg['solver_seconds_per_graph']))
        decoder.optimize_adjacency_matrix = optimise
        adj = decoder.decode_adjacency_matrix(encodings, predicted_prob_matrices=matrices,
                                              existence_threshold=decoder.existence_threshold)
    else:
        adj = threshold_adjacencies(encodings, matrices, cfg['edge_threshold'], decoder.existence_threshold)
    provider = pipeline.conditioning_to_node_embeddings_generator
    edge_labels = provider.predict_edge_labels(conditions, encodings, adj)
    graphs = decoder.decode(encodings, adj_mtx_list=adj, predicted_edge_labels_list=edge_labels,
                            predicted_prob_matrices=matrices, conditional_graph_encodings=conditions)
    return graphs


def run_audit(pipeline, references, requested, output, cfg, cycle_labels, leg_labels):
    """Synthetic paired controls/decoder audit of the newly trained model."""
    output = Path(output)
    if int(cfg['sampling_steps']) < 2:
        raise ValueError('Use at least two schedule levels; zero/one is not a valid no-diffusion baseline.')
    if not cfg['sampling_seeds'] or len(set(cfg['sampling_seeds'])) != len(cfg['sampling_seeds']):
        raise ValueError('Supply a nonempty list of distinct sampling seeds.')
    output.mkdir(parents=True, exist_ok=False)
    provider = pipeline.conditioning_to_node_embeddings_generator
    generator = provider.conditional_node_generator
    module = generator.model
    module.eval()
    before = weight_hash(module)
    for fixed, fitted in [('fixed_node_count', 'node_count_model'),
                          ('fixed_degree_histogram', 'degree_histogram_model'),
                          ('fixed_label_histogram', 'label_histogram_model')]:
        if getattr(generator, fixed, None) is None and getattr(generator, fitted, None) is None:
            raise ValueError(f'Fitted structural control is missing: {fitted}. Enable it before training.')
    dump_json(output / 'audit_config.json', cfg)
    rows, graph_records, pair_records = [], [], []
    conditions = np.asarray(pipeline.graph_encode(references))
    assert len(conditions) == len(references) == len(requested)
    np.save(output / 'heldout_conditions.npy', conditions)
    all_flags = dict(use_condition_node_count=True, use_condition_degree_histogram=True,
                     use_condition_label_histogram=True, total_steps=int(cfg['sampling_steps']))
    for seed in cfg['sampling_seeds']:
        for start in range(0, len(references), cfg['batch_size']):
            stop = min(start+cfg['batch_size'], len(references))
            c = conditions[start:stop]
            seed_all(int(seed)+100003*start)
            captured = {}
            raw_flags = {k: False for k in all_flags if k != 'total_steps'}
            raw_flags['total_steps'] = all_flags['total_steps']
            with overrides(generator, **raw_flags), capture_generation(module, captured):
                raw = generator.predict(c)
            if captured.get('calls') != 1:
                raise RuntimeError('Expected exactly one neural generation call per paired batch.')
            raw_p = provider.predict_edge_probabilities(c, raw)
            if raw_p is None:
                raise RuntimeError('The learned edge head is missing; this audit requires edge supervision.')
            with overrides(generator, **all_flags), replay_generation(module, captured):
                controlled = generator.predict(c)
            controlled_p = provider.predict_edge_probabilities(c, controlled)
            cached_scores = probability_hash(controlled_p)
            stages = [('neural_output', raw, raw_p, False),
                      ('plus_controls', controlled, controlled_p, False),
                      ('plus_constrained_decoder', controlled, controlled_p, True)]
            for name, x, p, constrained in stages:
                t = time.perf_counter()
                graphs = decode_stage(pipeline, c, x, p, constrained, cfg)
                if len(graphs) != stop-start:
                    raise RuntimeError('Decoder output count mismatch.')
                seconds = time.perf_counter()-t
                for j, graph in enumerate(graphs):
                    idx = start+j
                    metrics = graph_metrics(graph, references[idx], requested[idx], cycle_labels, leg_labels, x[j], cfg.get('leg_sizes', (4, 5)))
                    rows.append(dict(seed=seed, condition_index=idx, requested_cycle=requested[idx],
                                     variant=name, decode_batch_seconds=seconds, **metrics))
                    graph_records.append(dict(seed=seed, condition_index=idx, variant=name, graph=graph))
            assert probability_hash(controlled_p) == cached_scores
            pair_records.append(dict(seed=seed, first_condition=start, last_condition=stop-1,
                                     neural_generation_calls=1, replayed_identical_sample=True,
                                     neural_sample_sha256=hashlib.sha256(captured['tensor'].cpu().numpy().tobytes()).hexdigest(),
                                     controlled_edge_scores_sha256=cached_scores))
            pd.DataFrame(rows).to_csv(output / 'per_condition.csv', index=False)
            print(f'Seed {seed}: {stop}/{len(references)} paired conditions completed', flush=True)
    after = weight_hash(module)
    assert before == after, 'Evaluation changed neural model weights.'
    frame = pd.DataFrame(rows)
    assert len(frame) == len(references)*len(cfg['sampling_seeds'])*3
    assert not frame.duplicated(['seed', 'condition_index', 'variant']).any()
    per_seed = frame.groupby(['variant', 'seed'], as_index=False).agg(
        n=('strict', 'size'), strict_rate=('strict', 'mean'), cycle_rate=('exact_cycle', 'mean'),
        connected_rate=('connected', 'mean'), cycle_leg_shape_rate=('cycle_leg_shape', 'mean'),
        node_count_mae=('node_count_error', 'mean'), degree_hist_l1=('degree_hist_l1', 'mean'),
        label_hist_l1=('label_hist_l1', 'mean'), predicted_degree_mae=('predicted_degree_mae', 'mean'))
    summary = per_seed.groupby('variant').agg({c: ['mean', 'std'] for c in per_seed.columns if c not in ['variant', 'seed', 'n']})
    summary.columns = ['_'.join(c) for c in summary.columns]
    summary = summary.reset_index()
    by_cycle = frame.groupby(['variant', 'seed', 'requested_cycle'], as_index=False).agg(
        n=('strict', 'size'), strict_rate=('strict', 'mean'), cycle_rate=('exact_cycle', 'mean'), connected_rate=('connected', 'mean'))
    per_seed.to_csv(output / 'per_seed.csv', index=False)
    summary.to_csv(output / 'summary.csv', index=False)
    by_cycle.to_csv(output / 'by_cycle.csv', index=False)
    dump_json(output / 'pairing_checks.json', pair_records)
    dump_json(output / 'weight_check.json', dict(before=before, after=after, unchanged=before == after))
    import pickle
    with (output / 'decoded_graphs.pkl').open('wb') as f:
        pickle.dump(graph_records, f)
    return dict(per_condition=frame, per_seed=per_seed, summary=summary, by_cycle=by_cycle, output=output)


def show_report(result):
    import matplotlib.pyplot as plt
    order = ['neural_output', 'plus_controls', 'plus_constrained_decoder']
    s = result['summary'].set_index('variant').reindex(order)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    labels = ['Denoiser + heads', '+ controls', '+ constrained decoder']
    for ax, metric, title in zip(axes, ['strict_rate', 'connected_rate'], ['Strict conditional success', 'Connected graphs']):
        ax.bar(range(3), s[metric+'_mean']*100, yerr=s[metric+'_std'].fillna(0)*100,
               color=['#a6c8db', '#4b8bb4', '#245570'], capsize=3)
        ax.set_xticks(range(3), labels, rotation=12)
        ax.set_ylim(0, 105)
        ax.set_ylabel('%')
        ax.set_title(title)
        ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(result['output']/'paired_components.png', dpi=220, bbox_inches='tight')
    plt.show()
    return result['summary']


## Preflight and fresh training

Both run flags are already `True`. Execute the next cell once. Each execution creates a fresh timestamped run. The default baseline cap remains 5,000 epochs with validation early stopping (patience 100). A quick pilot can be configured in the earlier commented overrides, but must be reported as a pilot.

Before training, the cell checks source imports, requested device and CBC availability. Sparse encoding and SVD print progress. The original source's 10% internal validation split is retained. Held-out evaluation graphs remain separate. Condition scaling and controls use the training pool as in the source pipeline. The CUDA model starts only after CPU preparation and control fitting complete.

Epoch metrics are written to `training_logs/version_0/metrics.csv` and checkpoints to `checkpoints/` inside the new run. The progress bar is enabled. Configuration, source hashes, package versions and Git revision are saved. Keep this kernel alive for the paired audit; no previous-run resume path is provided.


In [ ]:
if RUN_TRAINING:
    import pulp
    from coco_grape.data_processor.generative.conditional_denoising_node_generator import ConditionalNodeGenerator
    from coco_grape.data_processor.generative.decompositional_encoder_decoder import DecompositionalEncoderDecoder
    assert pulp.PULP_CBC_CMD(msg=False).available(), 'CBC solver unavailable; check your PuLP/CBC installation.'
    MODEL, TEST_GRAPHS, TEST_TARGETS, RUN_DIR, TRAINING_RECEIPT = fit_fresh(
        PROJECT_ROOT, OUTPUT_ROOT, DATASET_CONFIG, GRAPH_CONFIG, NODE_CONFIG,
        DENOISE_CONFIG, DECODER_CONFIG, PIPELINE_CONFIG, TRAINER_CONFIG,
        training_seed=TRAINING_SEED, cpu_threads=CPU_THREADS,
        preparation_cfg=PREPARATION_CONFIG,
    )
    dump_json(RUN_DIR/'baseline_source_audit.json', BASELINE_AUDIT)
    display(pd.DataFrame([TRAINING_RECEIPT]))
else:
    print('Training disabled. Check the configuration, then enable RUN_TRAINING.')


## Paired evaluation

For each condition and sampling seed, the neural sampler runs **once**. Its tensor is captured and replayed to apply the three separately fitted controls, so changes cannot be explained by drawing a second neural sample. Stages 2 and 3 share their node features and learned edge-probability matrices. Their only intended difference is threshold adjacency versus constrained adjacency optimisation. Sampling-time categorical projections remain enabled in every stage, matching the thesis baseline. The graph decoder uses the same trained label interpretation in every stage.

The reported `exact_cycle` and `strict` definitions match the original evaluation cell: exactly one cycle of the requested length, plus role-valid labels for strict success. **They do not themselves guarantee connectedness or a single path attachment.** Connectedness and complete cycle–leg shape are therefore reported separately. All conditions, including unsuccessful decodes, stay in the denominator. Sampling seeds share one trained model; their variation is not training-seed uncertainty. With one sampling seed, no standard deviation is estimated.

`decode_batch_seconds` records decoding only and repeats the same batch duration on each row. **Do not sum this column over conditions or call it total runtime.** This experiment addresses attribution, not a runtime benchmark.


In [ ]:
if RUN_AUDIT:
    assert 'MODEL' in globals() and TRAINING_RECEIPT['from_scratch'], 'Train a new model in this kernel first.'
    REFERENCES, REQUESTED, HELDOUT_INDICES = choose_conditions(
        TEST_GRAPHS, TEST_TARGETS, DATASET_CONFIG['cycle_sizes'],
        AUDIT_CONFIG['per_cycle'], AUDIT_CONFIG['selection_seed'],
    )
    audit_dir = RUN_DIR / ('audit_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    RESULT = run_audit(MODEL, REFERENCES, REQUESTED, audit_dir, AUDIT_CONFIG,
                       DATASET_CONFIG['cycle_labels'], DATASET_CONFIG['leg_labels'])
    dump_json(audit_dir / 'heldout_indices.json', HELDOUT_INDICES)
    display(show_report(RESULT))
    display(RESULT['by_cycle'])
    print('Saved results:', audit_dir)
else:
    print('Evaluation disabled. After training, set RUN_AUDIT=True and run this cell.')


## What to report

Report fresh-training settings, training seed, number of held-out conditions, sampling seeds, and all three stage results. Use `summary.csv`, `per_seed.csv`, `by_cycle.csv`, and `paired_components.png`. `pairing_checks.json` records sample reuse and edge-score hashes; `weight_check.json` verifies frozen weights.

An improvement from stage 1 to 2 measures the **combined effect of the three fitted structural controls**, including how the learned edge head responds to changed node features. An improvement from stage 2 to 3 measures the decoder's effect with the same edge scores. These comparisons do not isolate individual controls or individual learned heads. Use fresh separately trained loss ablations for those questions. Do not interpret constraint-enforced connectedness as a property learned by diffusion.

Optional later experiment: repeat fresh training with a different training seed. A sweep over sampling steps only studies the sampler budget; it is not a diffusion-free comparison. Keep new results separate from the historical thesis table until you have examined them.


## Git and remote workflow

Commit the notebook itself. Generated runs default to `~/gcdg_runs`, outside the checkout; copy only selected compact reports into Git if needed. No Git command is executed by this notebook. Do not use `git reset --hard` or `git clean -fd` as routine synchronisation: they discard local edits/untracked files. Use `git status` and `git pull --ff-only` in a clean checkout.

On the local machine, after copying this notebook into `Denoise-generator/`:

```sh
git status
git add GCDG_From_Scratch_Component_Audit.ipynb
git commit -m "Make GCDG experiment portable with sparse preparation"
git push
```

On the remote machine, in the same repository:

```sh
git status
git pull --ff-only
# Activate your CUDA-enabled Python environment, then launch Jupyter from this checkout.
python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
jupyter lab
```

Open the notebook, check the configuration/device and run all cells. Both run flags are already enabled. For a remote terminal-only run, first ensure `nbconvert`, `ipykernel` and your CUDA environment are installed:

```sh
mkdir -p "$HOME/gcdg_runs/notebook_logs"
GCDG_DEVICE=cuda \
  jupyter nbconvert --execute --to notebook \
  --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3 \
  --output GCDG_executed.ipynb --output-dir "$HOME/gcdg_runs/notebook_logs" \
  GCDG_From_Scratch_Component_Audit.ipynb
```

Run this terminal command inside `tmux` or your remote scheduler if you need it to survive SSH disconnection. Make sure the selected Jupyter `python3` kernel is the CUDA environment. Set `GCDG_PROJECT_ROOT`, `GCDG_RUNS_DIR`, `GCDG_DEVICE`, or `GCDG_CPU_THREADS` to override the defaults. Keep scientific hyperparameters in the editable dictionaries above.
